In [31]:
import torch
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    total_loss, correct = 0.0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total_loss += loss_fn(out, y).item() * len(y)
            correct += (out.argmax(dim=1) == y).sum().item()

    n = len(loader.dataset)
    return total_loss / n, correct / n * 100


def train_one_epoch(model, loader, opt):
    model.train()

    # 고정된 BatchNorm은 학습 모드로 바꾸지 않음
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
            m.eval()

    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        loss = loss_fn(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item() * len(y)

    return total_loss / len(loader.dataset)


def fit(model, loader, epochs=5, opt=None):
    if opt is None:
        opt = torch.optim.SGD(model.parameters(), lr=0.1)

    train_losses, val_losses, val_accs = [], [], []

    for epoch in range(epochs):
        train_losses.append(train_one_epoch(model, loader, opt))
        val_loss, val_acc = evaluate(model, val_fast)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

    best = max(val_accs)
    best_epoch = val_accs.index(best) + 1
    print("best", best_epoch, round(best, 2),
          "last", round(val_accs[-1], 2))

    return train_losses, val_losses, val_accs

In [9]:
#준비 셀 1. CIFAR-10과 실험실
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset, TensorDataset
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True # cuDNN도 결정적 계산만
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225) # ImageNet 사진의 평균, 표준편차
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
train_all = datasets.CIFAR10(root="data", train=True, download=True, transform=tf)
g = torch.Generator().manual_seed(0) # 학습과 검증 분리, seed 0
train_set, val_set = random_split(train_all, [45000, 5000], generator=g)
lab = Subset(train_all, train_set.indices[:5000]) # 실험실 5,000장
xs = torch.stack([img for img, _ in lab])
ys = torch.tensor([y for _, y in lab])
xv = torch.stack([img for img, _ in val_set]) # 검증 5,000장
yv = torch.tensor([y for _, y in val_set])
lab_loader = DataLoader(TensorDataset(xs, ys), batch_size=64, shuffle=True)
val_fast = DataLoader(TensorDataset(xv, yv), batch_size=256, shuffle=False)
print(xs.shape, len(lab_loader), len(val_fast)) # → torch.Size([5000, 3, 32, 32]) 79 20

torch.Size([5000, 3, 32, 32]) 79 20


In [ ]:
#train_one_epoch의 앞부분, 더한 세 줄
def train_one_epoch(model, loader, opt):
 model.train()
 for m in model.modules(): # 모든 층을 하나씩 꺼내
    if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
        m.eval() # 고정한 BN은 평균, 분산 기록도 그대로
 ... # 아래는 지난시간과 같다

In [16]:
import time
import torch
import torch.nn as nn

def make_cnn(widths=(16, 32)): 
    """CNN 모델 생성 함수"""
    layers, cin, size = [], 3, 32
    for w in widths:
        layers += [
            nn.Conv2d(cin, w, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ]
        cin, size = w, size // 2  # 루프 안에서 입력 채널과 크기 업데이트
    
    # 루프가 끝난 후 Flatten과 Linear 레이어 추가
    layers += [nn.Flatten(), nn.Linear(cin * size * size, 10)]
    return nn.Sequential(*layers).to(device)

def count_train(model): 
    """학습 가능한 파라미터 수 계산"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def run(model, opt, loader=lab_loader, epochs=5): 
    """실험 실행 함수"""
    print("train", count_train(model), end=" ")
    t0 = time.time()
    
    # 주의: 여기서 fit 함수가 반드시 정의되어 있어야 합니다!
    result = fit(model, loader, epochs, opt) 
    
    
    print(" time", round(time.time() - t0, 1)) # 걸린 시간(초)
    return result


In [30]:
#예제 9-1. 지난시간 make_cnn으로 블록 3개, 처음부터 5 epoch
torch.manual_seed(0)
model = make_cnn((32, 64, 128)) # 블록 3개 CNN, 처음부터 학습
opt = torch.optim.AdamW(model.parameters(), lr=0.001)
result = run(model, opt)
# → (예) train 113738 best 4 54.84 last 54.24
# → (예) time 2.5

train 113738 best 4 55.36 last 53.2
 time 1.7


In [19]:
#예제 9-2. 사전학습 ResNet18로 CIFAR-10 사진 6장
from torchvision.models import resnet18, ResNet18_Weights
weights = ResNet18_Weights.DEFAULT # ImageNet으로 학습한 가중치
net = resnet18(weights=weights).to(device).eval()
names = weights.meta["categories"] # ImageNet의 1000가지 이름
print(net.fc)
up = nn.Upsample(size=224, mode="bilinear") # 32 × 32를 224 × 224로 키운다
with torch.no_grad():
 top = net(up(xv[:6]).to(device)).argmax(dim=1)
for i in range(6):
 print(train_all.classes[yv[i]], "→", names[top[i]])
# → Linear(in_features=512, out_features=1000, bias=True)
# → (예) truck → moving van
# → (예) cat → EntleBucher
# → (예) automobile → moving van
# → (예) frog → sidewinder
# → (예) automobile → chain saw
# → (예) cat → muzzle

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Linear(in_features=512, out_features=1000, bias=True)
truck → moving van
cat → EntleBucher
automobile → moving van
frog → sidewinder
automobile → chain saw
cat → muzzle


In [20]:
#준비 셀 4. 사전학습 모델 만들기
from torchvision.models import resnet18, ResNet18_Weights
def make_resnet(pretrained=True):
 weights = ResNet18_Weights.DEFAULT if pretrained else None
 net = resnet18(weights=weights) # 사전학습 ResNet18 (None이면 무작위)
 net.fc = nn.Linear(512, 10) # Classifier 교체, 1000 → 10
 up = nn.Upsample(size=112, mode="bilinear") # 사진을 112로 키워 넣는다
 model = nn.Sequential(up, net)
 return model.to(device), net # model 안에 net이 들어 있다

In [21]:
mnodel, net = make_resnet()
for name, layer in net.named_children():
    print(name, sum(p.numel() for p  in layer.parameters()))
print("total", count_train(model))

conv1 9408
bn1 128
relu 0
maxpool 0
layer1 147968
layer2 525568
layer3 2099712
layer4 8393728
avgpool 0
fc 5130
total 113738


In [22]:
#실습 1. 정답 예시
model, net = make_resnet()
for name, layer in net.named_children(): # 1. 큰 층마다 파라미터 수
 print(name, sum(p.numel() for p in layer.parameters()))
print("total", count_train(model))
# → conv1 9408
# → bn1 128
# → relu 0
# → maxpool 0
# → layer1 147968
# → layer2 525568
# → layer3 2099712
# → layer4 8393728
# → avgpool 0
# → fc 5130
# → total 11181642

conv1 9408
bn1 128
relu 0
maxpool 0
layer1 147968
layer2 525568
layer3 2099712
layer4 8393728
avgpool 0
fc 5130
total 11181642


In [25]:
#준비 셀 5. 어느 층부터 학습할지
ORDER = ["conv1", "bn1", "layer1", "layer2", "layer3", "layer4", "fc"] # 앞에서 뒤로
def unfreeze_from(net, start): # start 층과 그 뒤만 학습, 앞쪽은 Freeze
 for name, p in net.named_parameters():
    layer = name.split(".")[0] # "layer4.0.conv1.weight" → "layer4"
    p.requires_grad = ORDER.index(layer) >= ORDER.index(start)

In [29]:
#예제 9-3. fc만 학습 5 epoch
torch.manual_seed(0)
model, net = make_resnet()
unfreeze_from(net, "fc") # fc만 학습, 나머지는 Freeze
opt = torch.optim.AdamW(model.parameters(), lr=0.001)
result = run(model, opt)
# → (예) train 5130 best 4 79.86 last 79.82
# → (예) time 3.5

train 5130 best 4 79.86 last 79.82
 time 5.8


In [ ]:
# 실습
# 1. make_resnet(pretrained=False)로 같은 Feature Extraction 하기
# 2. 사전학습 없이 전체를 학습하고 아래 표에 적기
# 3. 사전학습 Feature Extraction과 무엇이 다른지 설명하기

In [32]:
#실습 2. 정답 예시
torch.manual_seed(0) # 1. 사전학습 없이 같은 방법
model, net = make_resnet(pretrained=False)
unfreeze_from(net, "fc")
opt = torch.optim.AdamW(model.parameters(), lr=0.001)
print("random fc", end=" ")
result = run(model, opt)
torch.manual_seed(0) # 2. 사전학습 없이 전체 학습
model, net = make_resnet(pretrained=False)
opt = torch.optim.AdamW(model.parameters(), lr=0.001)
print("random all", end=" ")
result = run(model, opt)
# → (예) random fc train 5130 best 4 23.48 last 19.82
# → (예) time 3.5
# → (예) random all train 11181642 best 5 51.04 last 51.04
# → (예) time 10.1

random fc train 5130 best 4 23.46 last 19.82
 time 5.8
random all train 11181642 best 4 49.72 last 49.4
 time 12.4
